# 🎙️ Train Wake Words: "Hey Money Penny" & "Hey Sheila"

This notebook trains two custom wake word models using [openWakeWord](https://github.com/dscripka/openWakeWord).

**What it does:**
1. Installs dependencies (Piper TTS, openWakeWord, etc.)
2. Downloads training datasets (RIRs, background audio, negative features)
3. Generates synthetic speech clips for each wake phrase
4. Augments clips with noise/reverb for robustness
5. Trains two models and exports as `.onnx` + `.tflite`

**Runtime:** ~60-90 min total on a T4 GPU (free Colab tier works fine)

**Output:** Two model files in Google Drive:
- `hey_money_penny.onnx`
- `hey_sheila.onnx`

---
⚡ **Instructions:** Runtime → Change runtime type → T4 GPU, then Run All (Ctrl+F9)

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

# ── Fix matplotlib backend (Colab sets MPLBACKEND to inline, which breaks venv subprocesses)
os.environ['MPLBACKEND'] = 'Agg'

# ── Python 3.11 venv ──────────────────────────────────────────────
# piper-phonemize only has Linux wheels for Python ≤3.11.
# Colab is now Python 3.12, so we install 3.11 and create a venv.
VENV = '/content/venv311'
VPYTHON = f'{VENV}/bin/python'
VPIP = f'{VENV}/bin/pip'

if not os.path.exists(VPYTHON):
    !add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
    !apt-get install -y -qq python3.11 python3.11-venv python3.11-dev espeak-ng > /dev/null 2>&1
    !python3.11 -m venv {VENV}
    # Bootstrap pip
    !{VPIP} install -q --upgrade pip
    print(f'✅ Python 3.11 venv created at {VENV}')
else:
    print(f'✅ Using existing venv at {VENV}')

# Ensure espeak-ng is available (needed by piper TTS phonemizer)
!dpkg -s espeak-ng > /dev/null 2>&1 || apt-get install -y -qq espeak-ng > /dev/null 2>&1

!{VPYTHON} --version

# ── Clone repos (idempotent) ──────────────────────────────────────
# Ensure we have dscripka's fork (not rhasspy's — different generate_samples signature)
if os.path.exists('piper-sample-generator'):
    remote = !cd piper-sample-generator && git remote get-url origin
    if 'dscripka' not in remote[0]:
        !rm -rf piper-sample-generator
        print('Removed rhasspy fork, will re-clone dscripka fork')
if not os.path.exists('piper-sample-generator'):
    !git clone -q https://github.com/dscripka/piper-sample-generator
# Download TTS model (save as the filename dscripka's fork expects as default)
!wget -q -nc -O piper-sample-generator/models/en-us-libritts-high.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

if not os.path.exists('openwakeword'):
    !git clone -q https://github.com/dscripka/openwakeword

# ── Install packages into the venv ────────────────────────────────
# Pin pyarrow + datasets first (newer pyarrow drops PyExtensionType)
!{VPIP} install -q pyarrow==14.0.2 datasets==2.14.6

# piper-phonemize (cp311 wheel from GitHub release)
!{VPIP} install -q \
    'https://github.com/rhasspy/piper-phonemize/releases/download/v1.1.0/piper_phonemize-1.1.0-cp311-cp311-manylinux_2_28_x86_64.whl'
!{VPIP} install -q webrtcvad

# piper-sample-generator deps (not a pip package — imported via sys.path in train.py)
!{VPIP} install -q -r ./piper-sample-generator/requirements.txt

# openWakeWord (editable install)
!{VPIP} install -q -e ./openwakeword

# Training dependencies — install torch first (GPU), then the rest
!{VPIP} install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!{VPIP} install -q scipy==1.11.4 mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
    speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
    acoustics==0.2.6 pronouncing==0.2.0 deep-phonemizer==0.0.19

# TFLite conversion (optional — .onnx is the primary output)
!{VPIP} install -q tensorflow-cpu tensorflow_probability onnx_tf 2>/dev/null || \
    echo '⚠️ TFLite deps skipped — .onnx models will still be produced'

# ── Download openWakeWord base models ─────────────────────────────
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
for name in ['embedding_model.onnx', 'embedding_model.tflite', 'melspectrogram.onnx', 'melspectrogram.tflite']:
    path = f'./openwakeword/openwakeword/resources/models/{name}'
    if not os.path.exists(path):
        !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{name} -O {path}

# Verify
!{VPYTHON} -c "import openwakeword; import piper_phonemize; print('✅ All imports OK')"
print('\n✅ Environment setup complete!')

In [ ]:
# Imports — run inside the venv
import os, subprocess
VENV = '/content/venv311'
VPYTHON = f'{VENV}/bin/python'

# Helper to run Python code in the venv
def vrun(code):
    """Run a Python snippet in the 3.11 venv"""
    result = subprocess.run([VPYTHON, '-c', code], capture_output=False)
    return result.returncode

In [ ]:
# Mount Google Drive early for dataset caching
# This avoids re-downloading ~20GB of data when the runtime restarts
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE = '/content/drive/MyDrive/wake_word_cache'
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f'📁 Drive cache: {DRIVE_CACHE}')

## 2. Download Training Datasets

This downloads:
- **MIT Room Impulse Responses** — for realistic reverb augmentation
- **Free Music Archive (FMA)** — background music/noise for augmentation
- **ACAV100M features** (16GB) — pre-computed negative examples (2,000 hrs of audio)
- **Validation features** — for false-positive rate estimation

⏱️ Takes ~10-15 min

In [ ]:
# Room Impulse Responses (MIT)
import os, shutil
VENV = '/content/venv311'
VPYTHON = f'{VENV}/bin/python'

rir_local = './mit_rirs'
rir_cache = os.path.join(DRIVE_CACHE, 'mit_rirs')

# Check Drive cache first
if os.path.exists(rir_cache) and len(os.listdir(rir_cache)) > 200:
    print(f'✅ Found {len(os.listdir(rir_cache))} RIR files in Drive cache — copying locally...')
    shutil.copytree(rir_cache, rir_local, dirs_exist_ok=True)
    print(f'   Copied to {rir_local}')
else:
    script = r'''
import os, numpy as np, datasets, scipy
from tqdm import tqdm
output_dir = './mit_rirs'
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
for row in tqdm(rir_dataset, desc='Downloading RIRs'):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print(f'RIR files: {len(os.listdir(output_dir))}')
'''
    with open('/tmp/download_rirs.py', 'w') as f:
        f.write(script)
    !{VPYTHON} /tmp/download_rirs.py

    # Save to Drive cache
    if os.path.exists(rir_local) and len(os.listdir(rir_local)) > 200:
        print('💾 Saving RIRs to Drive cache...')
        shutil.copytree(rir_local, rir_cache, dirs_exist_ok=True)
        print(f'   Cached {len(os.listdir(rir_cache))} files')

In [ ]:
# Background audio: Free Music Archive (FMA)
import os, shutil
VENV = '/content/venv311'
VPYTHON = f'{VENV}/bin/python'

fma_local = './fma'
fma_cache = os.path.join(DRIVE_CACHE, 'fma')

# Check Drive cache for FMA
fma_cached = os.path.exists(fma_cache) and len(os.listdir(fma_cache)) > 100

if fma_cached:
    print(f'✅ Found FMA ({len(os.listdir(fma_cache))} files) in Drive cache')
    shutil.copytree(fma_cache, fma_local, dirs_exist_ok=True)
    print('   Copied locally')
else:
    script = r'''
import os, numpy as np, datasets, scipy
from tqdm import tqdm

# FMA (Free Music Archive) — background music/noise for augmentation
output_dir = './fma'
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
n_hours = 4  # more hours since we're not using AudioSet
for i in tqdm(range(n_hours * 3600 // 30), desc='Downloading FMA'):
    try:
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    except StopIteration:
        break
print(f'FMA: {len(os.listdir(output_dir))} files')
'''
    with open('/tmp/download_background.py', 'w') as f:
        f.write(script)
    !{VPYTHON} /tmp/download_background.py

    # Save to Drive cache
    if os.path.exists(fma_local) and len(os.listdir(fma_local)) > 100:
        print('💾 Saving FMA to Drive cache...')
        shutil.copytree(fma_local, fma_cache, dirs_exist_ok=True)
        print('   Cached to Drive')

In [ ]:
# Pre-computed negative features (16GB) + validation features
import os, shutil

features = [
    ('openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
     'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'),
    ('validation_set_features.npy',
     'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'),
]

for fname, url in features:
    local_path = f'./{fname}'
    cache_path = os.path.join(DRIVE_CACHE, fname)

    if os.path.exists(local_path) and os.path.getsize(local_path) > 1000:
        print(f'✅ {fname} already exists locally')
    elif os.path.exists(cache_path) and os.path.getsize(cache_path) > 1000:
        print(f'✅ Found {fname} in Drive cache — copying locally...')
        shutil.copy2(cache_path, local_path)
        size_gb = os.path.getsize(local_path) / 1024**3
        print(f'   Copied ({size_gb:.1f} GB)')
    else:
        print(f'⬇️  Downloading {fname}...')
        !wget -q --show-progress '{url}' -O '{local_path}'
        # Save to Drive cache
        if os.path.exists(local_path) and os.path.getsize(local_path) > 1000:
            print(f'💾 Saving {fname} to Drive cache...')
            shutil.copy2(local_path, cache_path)

print('\n✅ Feature files ready')
!ls -lh *.npy

## 3. Configure & Train Models

We train two models sequentially:
1. **hey_money_penny** — activates Moneypenny (Dustin's assistant)
2. **hey_sheila** — activates Sheila (Jess's assistant)

Each model takes ~20-30 min to train on a T4 GPU.

### Training parameters (production quality)
- 50,000 positive + negative synthetic samples each
- 5,000 validation samples
- 50,000 training steps
- Target: accuracy ≥ 0.7, recall ≥ 0.5, FP rate ≤ 0.2/hr

In [ ]:
import os
VENV = '/content/venv311'
VPYTHON = f'{VENV}/bin/python'

# Background audio paths
background_paths = ['./fma']

# ═══════════════════════════════════════════════════
# CONFIGURE BOTH WAKE WORDS HERE
# ═══════════════════════════════════════════════════

WAKE_WORDS = [
    {
        'target_phrase': ['hey money penny'],
        'model_name': 'hey_money_penny',
        'output_dir': './model_hey_money_penny',
        # Phrases that sound similar but should NOT trigger activation
        'custom_negative_phrases': [
            'hey monopoly', 'hey many penny', 'a money penny',
            'pay money penny', 'hey honey penny', 'hey money bunny',
            'hey money plenty', 'hey money twenty'
        ],
    },
    {
        'target_phrase': ['hey sheila'],
        'model_name': 'hey_sheila',
        'output_dir': './model_hey_sheila',
        'custom_negative_phrases': [
            'hey sheela', 'hey shelly', 'hey shelia',
            'say sheila', 'hey sealer', 'hey shield',
            'hey she la', 'hey dealer'
        ],
    },
]

print(f'Will train {len(WAKE_WORDS)} models: {[w["model_name"] for w in WAKE_WORDS]}')

In [ ]:
# Generate YAML configs for both models
import yaml

config_template = yaml.load(
    open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader
)

for ww in WAKE_WORDS:
    config = config_template.copy()
    config['target_phrase'] = ww['target_phrase']
    config['model_name'] = ww['model_name']
    config['output_dir'] = ww['output_dir']
    config['custom_negative_phrases'] = ww.get('custom_negative_phrases', [])

    # Production-quality training params
    config['n_samples'] = 50000
    config['n_samples_val'] = 5000
    config['steps'] = 50000
    config['target_accuracy'] = 0.7
    config['target_recall'] = 0.5
    config['target_false_positives_per_hour'] = 0.2
    config['max_negative_weight'] = 1500

    # Speed up TTS generation — T4 GPU (16GB VRAM) handles large batches easily
    config['tts_batch_size'] = 200

    # Data paths
    config['background_paths'] = background_paths
    config['background_paths_duplication_rate'] = [1] * len(background_paths)
    config['false_positive_validation_data_path'] = './validation_set_features.npy'
    config['feature_data_files'] = {
        'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    }
    config['batch_n_per_class'] = {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50
    }
    config['piper_sample_generator_path'] = './piper-sample-generator'

    # Save config
    config_path = f'{ww["model_name"]}.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f)
    print(f'✅ Saved config: {config_path}')
    print(f'   Phrase: {ww["target_phrase"]}')
    print(f'   Samples: {config["n_samples"]} train / {config["n_samples_val"]} val')
    print(f'   Steps: {config["steps"]}')
    print()

### Train Model 1: `hey_money_penny`

Three steps: generate clips → augment → train

In [ ]:
# Step 1: Generate synthetic clips for "hey money penny"
# With Drive caching — survives runtime disconnects
# train.py has built-in resume: it counts existing clips and only generates the rest
import os, shutil, threading, time, glob

MODEL_NAME = 'hey_money_penny'
LOCAL_BASE = f'./model_{MODEL_NAME}/{MODEL_NAME}'
CACHE_BASE = os.path.join(DRIVE_CACHE, f'clips_{MODEL_NAME}')
SUBDIRS = ['positive_train', 'positive_test', 'negative_train', 'negative_test']

# Restore cached clips from Drive (train.py will skip generating ones that already exist)
total_restored = 0
for subdir in SUBDIRS:
    cache_sub = os.path.join(CACHE_BASE, subdir)
    local_sub = os.path.join(LOCAL_BASE, subdir)
    os.makedirs(local_sub, exist_ok=True)
    if os.path.exists(cache_sub):
        cached = [f for f in os.listdir(cache_sub) if f.endswith('.wav')]
        if cached:
            for f in cached:
                dst = os.path.join(local_sub, f)
                if not os.path.exists(dst):
                    shutil.copy2(os.path.join(cache_sub, f), dst)
                    total_restored += 1
    local_count = len([f for f in os.listdir(local_sub) if f.endswith('.wav')])
    print(f'  {subdir}: {local_count} clips')
if total_restored > 0:
    print(f'🔄 Restored {total_restored} clips from Drive cache')

# Background sync thread — copies new clips to Drive every 60s
_sync_stop = threading.Event()
def _sync_to_drive():
    while not _sync_stop.is_set():
        _sync_stop.wait(60)
        if _sync_stop.is_set():
            break
        try:
            synced = 0
            for subdir in SUBDIRS:
                cache_sub = os.path.join(CACHE_BASE, subdir)
                local_sub = os.path.join(LOCAL_BASE, subdir)
                if not os.path.exists(local_sub): continue
                os.makedirs(cache_sub, exist_ok=True)
                local_wavs = set(f for f in os.listdir(local_sub) if f.endswith('.wav'))
                cached_wavs = set(f for f in os.listdir(cache_sub) if f.endswith('.wav'))
                new_files = local_wavs - cached_wavs
                for f in new_files:
                    shutil.copy2(os.path.join(local_sub, f), os.path.join(cache_sub, f))
                synced += len(new_files)
            if synced > 0:
                print(f'💾 Synced {synced} new clips to Drive')
        except Exception as e:
            print(f'⚠️ Sync error: {e}')

sync_thread = threading.Thread(target=_sync_to_drive, daemon=True)
sync_thread.start()
print('🔄 Background Drive sync started (every 60s)')

# Run generation (train.py skips subdirs that already have enough clips)
!{VPYTHON} openwakeword/openwakeword/train.py --training_config hey_money_penny.yaml --generate_clips

# Final sync
_sync_stop.set()
sync_thread.join(timeout=5)
total_cached = 0
for subdir in SUBDIRS:
    cache_sub = os.path.join(CACHE_BASE, subdir)
    local_sub = os.path.join(LOCAL_BASE, subdir)
    if not os.path.exists(local_sub): continue
    os.makedirs(cache_sub, exist_ok=True)
    for f in os.listdir(local_sub):
        if f.endswith('.wav'):
            shutil.copy2(os.path.join(local_sub, f), os.path.join(cache_sub, f))
            total_cached += 1
print(f'✅ Final sync: {total_cached} clips cached to Drive')

In [ ]:
# Step 2: Augment clips with noise + reverb
# Cache augmented features to Drive too
import os, shutil, glob

AUG_CACHE = os.path.join(DRIVE_CACHE, 'augmented_hey_money_penny')
AUG_LOCAL = './model_hey_money_penny'

# Check if augmented .npy files already cached
cached_npy = glob.glob(os.path.join(AUG_CACHE, '*.npy')) if os.path.exists(AUG_CACHE) else []
local_npy = glob.glob(os.path.join(AUG_LOCAL, '*.npy'))

if cached_npy and not local_npy:
    print(f'🔄 Restoring {len(cached_npy)} augmented feature files from Drive...')
    os.makedirs(AUG_LOCAL, exist_ok=True)
    for f in cached_npy:
        shutil.copy2(f, os.path.join(AUG_LOCAL, os.path.basename(f)))
    local_npy = glob.glob(os.path.join(AUG_LOCAL, '*.npy'))
    print(f'   Restored {len(local_npy)} files — skipping augmentation')
else:
    !{VPYTHON} openwakeword/openwakeword/train.py --training_config hey_money_penny.yaml --augment_clips

    # Cache augmented features to Drive
    local_npy = glob.glob(os.path.join(AUG_LOCAL, '*.npy'))
    if local_npy:
        os.makedirs(AUG_CACHE, exist_ok=True)
        for f in local_npy:
            shutil.copy2(f, os.path.join(AUG_CACHE, os.path.basename(f)))
        print(f'💾 Cached {len(local_npy)} augmented feature files to Drive')

In [ ]:
# Step 3: Train the model
!{VPYTHON} openwakeword/openwakeword/train.py --training_config hey_money_penny.yaml --train_model
print('\n✅ hey_money_penny training complete!')

### Train Model 2: `hey_sheila`

In [ ]:
# Step 1: Generate synthetic clips for "hey sheila"
# With Drive caching — survives runtime disconnects
import os, shutil, threading, time

MODEL_NAME = 'hey_sheila'
LOCAL_BASE = f'./model_{MODEL_NAME}/{MODEL_NAME}'
CACHE_BASE = os.path.join(DRIVE_CACHE, f'clips_{MODEL_NAME}')
SUBDIRS = ['positive_train', 'positive_test', 'negative_train', 'negative_test']

# Restore cached clips from Drive
total_restored = 0
for subdir in SUBDIRS:
    cache_sub = os.path.join(CACHE_BASE, subdir)
    local_sub = os.path.join(LOCAL_BASE, subdir)
    os.makedirs(local_sub, exist_ok=True)
    if os.path.exists(cache_sub):
        cached = [f for f in os.listdir(cache_sub) if f.endswith('.wav')]
        if cached:
            for f in cached:
                dst = os.path.join(local_sub, f)
                if not os.path.exists(dst):
                    shutil.copy2(os.path.join(cache_sub, f), dst)
                    total_restored += 1
    local_count = len([f for f in os.listdir(local_sub) if f.endswith('.wav')])
    print(f'  {subdir}: {local_count} clips')
if total_restored > 0:
    print(f'🔄 Restored {total_restored} clips from Drive cache')

# Background sync thread
_sync_stop2 = threading.Event()
def _sync_to_drive2():
    while not _sync_stop2.is_set():
        _sync_stop2.wait(60)
        if _sync_stop2.is_set():
            break
        try:
            synced = 0
            for subdir in SUBDIRS:
                cache_sub = os.path.join(CACHE_BASE, subdir)
                local_sub = os.path.join(LOCAL_BASE, subdir)
                if not os.path.exists(local_sub): continue
                os.makedirs(cache_sub, exist_ok=True)
                local_wavs = set(f for f in os.listdir(local_sub) if f.endswith('.wav'))
                cached_wavs = set(f for f in os.listdir(cache_sub) if f.endswith('.wav'))
                new_files = local_wavs - cached_wavs
                for f in new_files:
                    shutil.copy2(os.path.join(local_sub, f), os.path.join(cache_sub, f))
                synced += len(new_files)
            if synced > 0:
                print(f'💾 Synced {synced} new clips to Drive')
        except Exception as e:
            print(f'⚠️ Sync error: {e}')

sync_thread2 = threading.Thread(target=_sync_to_drive2, daemon=True)
sync_thread2.start()
print('🔄 Background Drive sync started (every 60s)')

!{VPYTHON} openwakeword/openwakeword/train.py --training_config hey_sheila.yaml --generate_clips

# Final sync
_sync_stop2.set()
sync_thread2.join(timeout=5)
total_cached = 0
for subdir in SUBDIRS:
    cache_sub = os.path.join(CACHE_BASE, subdir)
    local_sub = os.path.join(LOCAL_BASE, subdir)
    if not os.path.exists(local_sub): continue
    os.makedirs(cache_sub, exist_ok=True)
    for f in os.listdir(local_sub):
        if f.endswith('.wav'):
            shutil.copy2(os.path.join(local_sub, f), os.path.join(cache_sub, f))
            total_cached += 1
print(f'✅ Final sync: {total_cached} clips cached to Drive')

In [ ]:
# Step 2: Augment clips
import os, shutil, glob

AUG_CACHE = os.path.join(DRIVE_CACHE, 'augmented_hey_sheila')
AUG_LOCAL = './model_hey_sheila'

cached_npy = glob.glob(os.path.join(AUG_CACHE, '*.npy')) if os.path.exists(AUG_CACHE) else []
local_npy = glob.glob(os.path.join(AUG_LOCAL, '*.npy'))

if cached_npy and not local_npy:
    print(f'🔄 Restoring {len(cached_npy)} augmented feature files from Drive...')
    os.makedirs(AUG_LOCAL, exist_ok=True)
    for f in cached_npy:
        shutil.copy2(f, os.path.join(AUG_LOCAL, os.path.basename(f)))
    print(f'   Restored — skipping augmentation')
else:
    !{VPYTHON} openwakeword/openwakeword/train.py --training_config hey_sheila.yaml --augment_clips

    local_npy = glob.glob(os.path.join(AUG_LOCAL, '*.npy'))
    if local_npy:
        os.makedirs(AUG_CACHE, exist_ok=True)
        for f in local_npy:
            shutil.copy2(f, os.path.join(AUG_CACHE, os.path.basename(f)))
        print(f'💾 Cached {len(local_npy)} augmented feature files to Drive')

In [ ]:
# Step 3: Train
!{VPYTHON} openwakeword/openwakeword/train.py --training_config hey_sheila.yaml --train_model
print('\n✅ hey_sheila training complete!')

## 4. Export Models to Google Drive

Save the trained `.onnx` models to your Drive for easy download.

In [ ]:
# Drive is already mounted from the cache step above
import shutil, os

output_drive = '/content/drive/MyDrive/wake_word_models'
os.makedirs(output_drive, exist_ok=True)

models_found = []
for ww in WAKE_WORDS:
    name = ww['model_name']
    out_dir = ww['output_dir']
    for ext in ['.onnx', '.tflite']:
        src = os.path.join(out_dir, f'{name}{ext}')
        if os.path.exists(src):
            dst = os.path.join(output_drive, f'{name}{ext}')
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1024**2
            print(f'✅ Saved: {dst} ({size_mb:.1f} MB)')
            models_found.append(dst)
        else:
            print(f'⚠️ Not found: {src}')

# Also save the configs for reference
for ww in WAKE_WORDS:
    shutil.copy2(f'{ww["model_name"]}.yaml', os.path.join(output_drive, f'{ww["model_name"]}.yaml'))

print(f'\n🎉 Done! {len(models_found)} model files saved to Google Drive: {output_drive}')

## 5. Quick Test (Optional)

Load the models and verify they work with openWakeWord.

In [ ]:
import os
VENV = '/content/venv311'
VPYTHON = f'{VENV}/bin/python'

# Write a quick test script
test_script = '''
import sys, os, numpy as np
from openwakeword.model import Model

models = [
    ('hey_money_penny', './model_hey_money_penny'),
    ('hey_sheila', './model_hey_sheila'),
]
for name, out_dir in models:
    model_path = os.path.join(out_dir, f'{name}.onnx')
    if os.path.exists(model_path):
        owwModel = Model(wakeword_models=[model_path])
        silence = np.zeros(1280, dtype=np.int16)
        for _ in range(5):
            prediction = owwModel.predict(silence)
        print(f'{name}: loaded OK, inference working')
        print(f'  Prediction on silence: {prediction}')
    else:
        print(f'{name}: model file not found at {model_path}')
'''
with open('/tmp/test_models.py', 'w') as f:
    f.write(test_script)

!{VPYTHON} /tmp/test_models.py
print('\n🎙️ Models are ready! Download from Google Drive → wake_word_models/')